# Experiment 005 - 28 Aug 2026

By  R.R.Rosa and G.Neri-Silva

COPDT-INPE-MCTI     +55 12 98129-8988

In [ ]:
# ============================================================
# SIMULATION V3 — FINAL CONSOLIDATED NUMERICAL EXPERIMENT
#
# Weak localized perturbations of a propagated Gaussian laser beam
#
# Scientific questions
# ------------------------------------------------------------
# 1. How strongly does a localized phase perturbation drive the
#    remotely observed transverse laser profile away from its
#    unperturbed Gaussian state?
#
# 2. How does this departure evolve with perturbation strength?
#
# 3. Are the residual fluctuations Gaussian or non-Gaussian?
#
# 4. Which probability model best describes the residuals?
#
# 5. Do the residuals possess a characteristic spatial
#    correlation length?
#
# Main outputs
# ------------------------------------------------------------
# fig2.png / fig2.pdf
# fig3.png / fig3.pdf
#
# diagnostic_deltaI.png
# diagnostic_residual_pdfs.png
#
# numerical_summary.csv
# realization_statistics.csv
# model_selection_by_realization.csv
# model_selection_summary.csv
# autocorrelation_curves.csv
# autocorrelation_summary.csv
#
# These CSV files support supplementary Cullen-Frey and
# spatial-correlation analyses.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter
from scipy.optimize import curve_fit
from scipy.stats import (
    skew,
    kurtosis,
    norm,
    t,
    gennorm
)

import scipy
import matplotlib


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

MASTER_SEED = 2026

rng_master = np.random.default_rng(
    MASTER_SEED
)


# ============================================================
# 3. NUMERICAL AND OPTICAL PARAMETERS
# ============================================================

# ------------------------------------------------------------
# Computational grid
# ------------------------------------------------------------

N = 1024

Lx = 8.0e-2          # transverse window [m] = 80 mm

x = np.linspace(
    -Lx/2,
    Lx/2,
    N
)

y = np.linspace(
    -Lx/2,
    Lx/2,
    N
)

dx = x[1] - x[0]
dy = y[1] - y[0]

X, Y = np.meshgrid(
    x,
    y,
    indexing="xy"
)


# ------------------------------------------------------------
# Laser parameters
# ------------------------------------------------------------

w0 = 1.0e-2          # input Gaussian beam parameter [m] = 10 mm

wavelength = 532e-9 # wavelength [m]

zprop = 0.60         # propagation distance [m]


# ------------------------------------------------------------
# Thermal-like phase perturbation
# ------------------------------------------------------------

# IMPORTANT:
# sigma is expressed in GRID PIXELS.
#
# We retain the original value for continuity with previous
# simulations. Its physical smoothing scale is printed below.

FILTER_SIGMA_PIXELS = 18.0

# Broad Gaussian perturbation envelope retained from original model
pz = 1.07e-1         # [m] = 107 mm


# Perturbation strengths
PHI_VALUES = np.array([
    0.00,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50
])


# Independent random realizations per non-zero phi0
N_REALIZATIONS = 100


# ------------------------------------------------------------
# Transverse-profile extraction
# ------------------------------------------------------------

# Average over 9 rows centred on beam axis
STRIP_HALF_WIDTH = 4


# ------------------------------------------------------------
# Correlation-length definition
# ------------------------------------------------------------

CORRELATION_THRESHOLD = np.exp(-1.0)


# ============================================================
# 4. BASIC NUMERICAL INFORMATION
# ============================================================

print("============================================================")
print("SIMULATION PARAMETERS")
print("============================================================")

print(f"Grid                      : {N} x {N}")
print(f"Window                    : {Lx*1e3:.3f} mm")
print(f"Grid spacing dx           : {dx*1e6:.3f} micrometres")

print(
    f"Phase smoothing sigma     : "
    f"{FILTER_SIGMA_PIXELS:.1f} pixels "
    f"(~{FILTER_SIGMA_PIXELS*dx*1e3:.3f} mm)"
)

print(f"Wavelength                : {wavelength*1e9:.1f} nm")
print(f"Input beam parameter w0   : {w0*1e3:.3f} mm")
print(f"Propagation distance      : {zprop:.3f} m")
print(f"Perturbation envelope pz  : {pz*1e3:.3f} mm")
print(f"Realizations / phi        : {N_REALIZATIONS}")
print()


# ============================================================
# 5. INPUT GAUSSIAN BEAM
# ============================================================

A0 = np.exp(
    -(X**2 + Y**2)
    / w0**2
)


# ============================================================
# 6. FRESNEL PROPAGATOR
# ============================================================

fx = np.fft.fftfreq(
    N,
    d=dx
)

fy = np.fft.fftfreq(
    N,
    d=dy
)

FX, FY = np.meshgrid(
    fx,
    fy,
    indexing="xy"
)

H_FRESNEL = np.exp(
    -1j
    * np.pi
    * wavelength
    * zprop
    * (FX**2 + FY**2)
)


def propagate(field):
    """
    Fresnel propagation using the transfer-function method.
    """

    return np.fft.ifft2(
        np.fft.fft2(field)
        * H_FRESNEL
    )


# ============================================================
# 7. OFF REFERENCE
# ============================================================

U_off = propagate(
    A0
)

I_off_raw = np.abs(
    U_off
)**2


# ------------------------------------------------------------
# COMMON NORMALIZATION
#
# Every OFF/ON image is normalized to the SAME reference:
# the maximum intensity of the OFF field.
# ------------------------------------------------------------

I_REFERENCE = np.max(
    I_off_raw
)

I_off = (
    I_off_raw
    / I_REFERENCE
)


# ============================================================
# 8. CORRELATED THERMAL-LIKE PHASE SCREEN
# ============================================================

def generate_phase_screen(
    phi0,
    rng
):
    """
    Generate one independent spatially correlated phase screen.
    """

    if np.isclose(
        phi0,
        0.0
    ):
        return np.zeros(
            (N, N)
        )

    noise = rng.normal(
        loc=0.0,
        scale=1.0,
        size=(N, N)
    )

    eta = gaussian_filter(
        noise,
        sigma=FILTER_SIGMA_PIXELS,
        mode="reflect"
    )

    eta = (
        eta
        - np.mean(eta)
    )

    sigma_eta = np.std(
        eta
    )

    M = np.exp(
        -(X**2 + Y**2)
        / pz**2
    )

    phase = (
        phi0
        * M
        * eta
        / sigma_eta
    )

    return phase


# ============================================================
# 9. SIMULATE ONE PERTURBED IMAGE
# ============================================================

def simulate_on(
    phi0,
    rng
):
    """
    Apply one phase perturbation and propagate the beam.
    """

    phase = generate_phase_screen(
        phi0,
        rng
    )

    A_on = (
        A0
        * np.exp(
            1j * phase
        )
    )

    U_on = propagate(
        A_on
    )

    I_on_raw = (
        np.abs(U_on)**2
    )

    I_on = (
        I_on_raw
        / I_REFERENCE
    )

    return (
        I_on,
        phase
    )


# ============================================================
# 10. TRANSVERSE PROFILE
# ============================================================

def transverse_profile(
    image,
    half_width=STRIP_HALF_WIDTH
):
    """
    Horizontal transverse profile averaged over a narrow
    strip centred on the beam axis.
    """

    ny, nx = image.shape

    centre_y = (
        ny // 2
    )

    strip = image[
        centre_y-half_width:
        centre_y+half_width+1,
        :
    ]

    return np.mean(
        strip,
        axis=0
    )


# ============================================================
# 11. GAUSSIAN PROFILE MODEL
# ============================================================

def gaussian_profile(
    xcoord,
    amplitude,
    x0,
    width,
    background
):
    """
    Gaussian transverse intensity profile:

    I(x) = A exp[-2 (x-x0)^2/w^2] + B
    """

    return (
        amplitude
        * np.exp(
            -2.0
            * (xcoord - x0)**2
            / width**2
        )
        + background
    )


# ============================================================
# 12. GAUSSIAN FIT AND DEPARTURE METRIC
# ============================================================

def fit_gaussian(
    profile
):
    """
    Fit Gaussian transverse profile and calculate residuals
    and normalized Gaussian-departure metric epsilon_G.
    """

    amplitude_guess = np.max(
        profile
    )

    x0_guess = x[
        np.argmax(
            profile
        )
    ]

    width_guess = w0

    background_guess = np.min(
        profile
    )

    p0 = [
        amplitude_guess,
        x0_guess,
        width_guess,
        background_guess
    ]

    lower_bounds = [
        0.0,
        -Lx/4,
        1e-4,
        -0.1
    ]

    upper_bounds = [
        2.0,
        Lx/4,
        Lx/2,
        0.2
    ]

    try:

        popt, _ = curve_fit(
            gaussian_profile,
            x,
            profile,
            p0=p0,
            bounds=(
                lower_bounds,
                upper_bounds
            ),
            maxfev=30000
        )

        fitted = gaussian_profile(
            x,
            *popt
        )

    except Exception:

        popt = np.array([
            np.nan,
            np.nan,
            np.nan,
            np.nan
        ])

        fitted = np.full_like(
            profile,
            np.nan
        )

    residual = (
        profile
        - fitted
    )

    if np.any(
        ~np.isfinite(residual)
    ):

        epsilon_G = np.nan

    else:

        denominator = np.sum(
            profile**2
        )

        epsilon_G = np.sqrt(
            np.sum(
                residual**2
            )
            / denominator
        )

    return (
        fitted,
        popt,
        residual,
        epsilon_G
    )


# ============================================================
# 13. STANDARDIZED RESIDUAL
# ============================================================

def standardize_residual(
    residual
):
    """
    Centre residual and divide by its standard deviation.
    """

    r = np.asarray(
        residual,
        dtype=float
    )

    r = r[
        np.isfinite(r)
    ]

    if len(r) < 10:
        return None

    r = (
        r
        - np.mean(r)
    )

    sigma = np.std(
        r,
        ddof=1
    )

    if sigma < 1e-14:
        return None

    return (
        r / sigma
    )


# ============================================================
# 14. NORMALIZED AUTOCORRELATION
# ============================================================

def normalized_autocorrelation(
    residual
):
    """
    Positive-lag normalized autocorrelation of a 1-D residual.
    """

    r = np.asarray(
        residual,
        dtype=float
    )

    r = (
        r
        - np.mean(r)
    )

    if np.std(r) < 1e-14:
        return None

    corr = np.correlate(
        r,
        r,
        mode="full"
    )

    corr = corr[
        len(r)-1:
    ]

    if corr[0] == 0:
        return None

    corr = (
        corr
        / corr[0]
    )

    return corr


# ============================================================
# 15. CORRELATION LENGTH
# ============================================================

def correlation_length(
    rho
):
    """
    Estimate correlation length at first crossing of 1/e.

    Linear interpolation is used between adjacent lag samples.
    """

    if rho is None:
        return np.nan

    threshold = (
        CORRELATION_THRESHOLD
    )

    below = np.where(
        rho <= threshold
    )[0]

    if len(below) == 0:
        return np.nan

    i = below[0]

    if i == 0:
        return 0.0

    y1 = rho[i-1]
    y2 = rho[i]

    x1 = (
        (i-1)
        * dx
    )

    x2 = (
        i
        * dx
    )

    if np.isclose(
        y2,
        y1
    ):
        lc = x2

    else:

        lc = (
            x1
            +
            (threshold-y1)
            * (x2-x1)
            / (y2-y1)
        )

    return (
        lc * 1e3
    )  # mm


# ============================================================
# 16. PROBABILITY MODEL FITTING
#
# Candidate models for signed residuals:
#
# Normal
# Student-t
# Generalized Normal
# ============================================================

def information_criteria(
    logL,
    n_params,
    n
):
    """
    AIC and BIC.
    """

    AIC = (
        2*n_params
        - 2*logL
    )

    BIC = (
        n_params*np.log(n)
        - 2*logL
    )

    return (
        AIC,
        BIC
    )


def fit_probability_models(
    data
):
    """
    Fit candidate distributions to ONE realization.

    This is intentionally performed realization-by-realization
    instead of concatenating all profiles into one enormous
    pseudo-independent sample.
    """

    output = {}


    # --------------------------------------------------------
    # Normal
    # --------------------------------------------------------

    params_n = norm.fit(
        data
    )

    logpdf = norm.logpdf(
        data,
        *params_n
    )

    logL = np.sum(
        logpdf
    )

    AIC, BIC = information_criteria(
        logL,
        2,
        len(data)
    )

    output["Normal"] = {
        "AIC": AIC,
        "BIC": BIC,
        "parameters": params_n
    }


    # --------------------------------------------------------
    # Student-t
    # --------------------------------------------------------

    params_t = t.fit(
        data
    )

    logpdf = t.logpdf(
        data,
        *params_t
    )

    logL = np.sum(
        logpdf
    )

    AIC, BIC = information_criteria(
        logL,
        3,
        len(data)
    )

    output["Student-t"] = {
        "AIC": AIC,
        "BIC": BIC,
        "parameters": params_t
    }


    # --------------------------------------------------------
    # Generalized Normal
    # --------------------------------------------------------

    params_g = gennorm.fit(
        data
    )

    logpdf = gennorm.logpdf(
        data,
        *params_g
    )

    logL = np.sum(
        logpdf
    )

    AIC, BIC = information_criteria(
        logL,
        3,
        len(data)
    )

    output["Generalized Normal"] = {
        "AIC": AIC,
        "BIC": BIC,
        "parameters": params_g
    }

    return output


# ============================================================
# 17. OFF GAUSSIAN REFERENCE
# ============================================================

profile_off = transverse_profile(
    I_off
)

(
    fit_off,
    params_off,
    residual_off,
    epsilon_off
) = fit_gaussian(
    profile_off
)


print("============================================================")
print("OFF GAUSSIAN REFERENCE")
print("============================================================")

print(
    f"epsilon_G                = "
    f"{epsilon_off:.8e}"
)

print(
    f"Gaussian amplitude       = "
    f"{params_off[0]:.8f}"
)

print(
    f"Gaussian centre          = "
    f"{params_off[1]*1e3:.6f} mm"
)

print(
    f"Gaussian width           = "
    f"{params_off[2]*1e3:.6f} mm"
)

print(
    f"Background               = "
    f"{params_off[3]:.8e}"
)

print()


# ============================================================
# 18. REPRESENTATIVE IMAGE FOR FIGURE 2
# ============================================================

PHI_REPRESENTATIVE = 0.50

rng_rep = np.random.default_rng(
    MASTER_SEED + 999
)

(
    I_on_rep,
    phase_rep
) = simulate_on(
    PHI_REPRESENTATIVE,
    rng_rep
)

profile_on_rep = transverse_profile(
    I_on_rep
)

(
    fit_on_rep,
    params_on_rep,
    residual_on_rep,
    epsilon_on_rep
) = fit_gaussian(
    profile_on_rep
)


# ============================================================
# 19. FIGURE 2
#
# Improved optical phenomenology
# ============================================================

fig2, axes = plt.subplots(
    2,
    2,
    figsize=(11.5, 8.2)
)

extent_mm = [
    x.min()*1e3,
    x.max()*1e3,
    y.min()*1e3,
    y.max()*1e3
]


# ------------------------------------------------------------
# COMMON COLOR SCALE
# ------------------------------------------------------------

vmax = max(
    np.max(I_off),
    np.max(I_on_rep)
)


# ------------------------------------------------------------
# (a) OFF
# ------------------------------------------------------------

im0 = axes[0,0].imshow(
    I_off,
    origin="lower",
    extent=extent_mm,
    vmin=0,
    vmax=vmax,
    aspect="equal"
)

axes[0,0].set_title(
    "(a) OFF"
)

axes[0,0].set_xlabel(
    "x (mm)"
)

axes[0,0].set_ylabel(
    "y (mm)"
)

fig2.colorbar(
    im0,
    ax=axes[0,0],
    label="Normalized intensity"
)


# ------------------------------------------------------------
# (b) Representative ON
# ------------------------------------------------------------

im1 = axes[0,1].imshow(
    I_on_rep,
    origin="lower",
    extent=extent_mm,
    vmin=0,
    vmax=vmax,
    aspect="equal"
)

axes[0,1].set_title(
    rf"(b) ON, $\phi_0={PHI_REPRESENTATIVE:.2f}$ rad"
)

axes[0,1].set_xlabel(
    "x (mm)"
)

axes[0,1].set_ylabel(
    "y (mm)"
)

fig2.colorbar(
    im1,
    ax=axes[0,1],
    label="Normalized intensity"
)


# ------------------------------------------------------------
# (c) Transverse profiles
# ------------------------------------------------------------

axes[1,0].plot(
    x*1e3,
    profile_off,
    linewidth=1.8,
    label="OFF profile"
)

axes[1,0].plot(
    x*1e3,
    fit_off,
    linestyle="--",
    linewidth=1.4,
    label="OFF Gaussian fit"
)

axes[1,0].plot(
    x*1e3,
    profile_on_rep,
    linewidth=1.8,
    label="ON profile"
)

axes[1,0].plot(
    x*1e3,
    fit_on_rep,
    linestyle="--",
    linewidth=1.4,
    label="ON Gaussian fit"
)

axes[1,0].set_title(
    "(c) Transverse beam profiles"
)

axes[1,0].set_xlabel(
    "x (mm)"
)

axes[1,0].set_ylabel(
    "Normalized intensity"
)

axes[1,0].legend(
    frameon=False,
    fontsize=8
)

axes[1,0].grid(
    alpha=0.20
)


# ------------------------------------------------------------
# (d) Residual
# ------------------------------------------------------------

axes[1,1].plot(
    x*1e3,
    residual_off,
    linewidth=1.3,
    label="OFF residual"
)

axes[1,1].plot(
    x*1e3,
    residual_on_rep,
    linewidth=1.5,
    label="ON residual"
)

axes[1,1].axhline(
    0,
    linestyle="--",
    linewidth=1
)

axes[1,1].set_title(
    "(d) Departure from best-fit Gaussian"
)

axes[1,1].set_xlabel(
    "x (mm)"
)

axes[1,1].set_ylabel(
    r"$I-I_G$"
)

axes[1,1].legend(
    frameon=False,
    fontsize=8
)

axes[1,1].grid(
    alpha=0.20
)


fig2.tight_layout()

fig2.savefig(
    "fig2.png",
    dpi=600,
    bbox_inches="tight"
)

fig2.savefig(
    "fig2.pdf",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 20. DIAGNOSTIC SPATIAL DIFFERENCE
# ============================================================

delta_I = (
    I_on_rep
    - I_off
)

absmax_delta = np.max(
    np.abs(delta_I)
)

plt.figure(
    figsize=(6.4, 5.2)
)

plt.imshow(
    delta_I,
    origin="lower",
    extent=extent_mm,
    aspect="equal",
    vmin=-absmax_delta,
    vmax=absmax_delta
)

plt.colorbar(
    label=r"$I_{\mathrm{ON}}-I_{\mathrm{OFF}}$"
)

plt.xlabel(
    "x (mm)"
)

plt.ylabel(
    "y (mm)"
)

plt.title(
    "Spatial intensity redistribution"
)

plt.tight_layout()

plt.savefig(
    "diagnostic_deltaI.png",
    dpi=400,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 21. STORAGE STRUCTURES
# ============================================================

realization_records = []

model_records = []

acf_records = []

residual_examples = {}


# ============================================================
# 22. PERTURBATION SWEEP
# ============================================================

for phi0 in PHI_VALUES:

    print()
    print("============================================================")
    print(f"PERTURBATION phi0 = {phi0:.3f} rad")
    print("============================================================")


    # --------------------------------------------------------
    # OFF is deterministic in this minimal model.
    #
    # We therefore include one OFF reference realization.
    #
    # Non-zero phase levels contain independent phase screens.
    # --------------------------------------------------------

    if np.isclose(
        phi0,
        0.0
    ):

        realization_profiles = [
            (
                0,
                profile_off
            )
        ]

    else:

        realization_profiles = []

        for realization_id in range(
            N_REALIZATIONS
        ):

            seed = int(
                rng_master.integers(
                    0,
                    2**32 - 1
                )
            )

            rng = np.random.default_rng(
                seed
            )

            image, phase = simulate_on(
                phi0,
                rng
            )

            profile = transverse_profile(
                image
            )

            realization_profiles.append(
                (
                    realization_id,
                    profile
                )
            )


    # --------------------------------------------------------
    # Analyse realizations
    # --------------------------------------------------------

    for (
        realization_id,
        profile
    ) in realization_profiles:

        (
            fitted,
            params,
            residual,
            epsilon_G
        ) = fit_gaussian(
            profile
        )

        if not np.isfinite(
            epsilon_G
        ):
            continue


        standardized_residual = standardize_residual(
            residual
        )

        if standardized_residual is None:
            continue


        # ----------------------------------------------------
        # Higher-order residual statistics
        # ----------------------------------------------------

        S_value = skew(
            standardized_residual,
            bias=False
        )

        K_value = kurtosis(
            standardized_residual,
            fisher=False,
            bias=False
        )


        # ----------------------------------------------------
        # Spatial autocorrelation
        # ----------------------------------------------------

        rho = normalized_autocorrelation(
            residual
        )

        lc_mm = correlation_length(
            rho
        )


        # ----------------------------------------------------
        # Save per-realization statistics
        # ----------------------------------------------------

        realization_records.append(
            {
                "phi0":
                    phi0,

                "realization":
                    realization_id,

                "epsilon_G":
                    epsilon_G,

                "skewness":
                    S_value,

                "skewness_squared":
                    S_value**2,

                "pearson_kurtosis":
                    K_value,

                "gaussian_amplitude":
                    params[0],

                "gaussian_centre_mm":
                    params[1]*1e3,

                "gaussian_width_mm":
                    params[2]*1e3,

                "gaussian_background":
                    params[3],

                "correlation_length_mm":
                    lc_mm
            }
        )


        # ----------------------------------------------------
        # Save complete normalized ACF
        # ----------------------------------------------------

        if rho is not None:

            max_lags_to_save = min(
                400,
                len(rho)
            )

            for lag_index in range(
                max_lags_to_save
            ):

                acf_records.append(
                    {
                        "phi0":
                            phi0,

                        "realization":
                            realization_id,

                        "lag_index":
                            lag_index,

                        "lag_mm":
                            lag_index
                            * dx
                            * 1e3,

                        "rho":
                            rho[
                                lag_index
                            ]
                    }
                )


        # ----------------------------------------------------
        # Model selection PER realization
        # ----------------------------------------------------

        if phi0 > 0:

            fitted_models = fit_probability_models(
                standardized_residual
            )

            min_AIC = min(
                value["AIC"]
                for value
                in fitted_models.values()
            )

            min_BIC = min(
                value["BIC"]
                for value
                in fitted_models.values()
            )


            for (
                model_name,
                result
            ) in fitted_models.items():

                params_model = result[
                    "parameters"
                ]

                record = {
                    "phi0":
                        phi0,

                    "realization":
                        realization_id,

                    "model":
                        model_name,

                    "AIC":
                        result["AIC"],

                    "BIC":
                        result["BIC"],

                    "Delta_AIC":
                        result["AIC"]
                        - min_AIC,

                    "Delta_BIC":
                        result["BIC"]
                        - min_BIC
                }


                # --------------------------------------------
                # Save interpretable shape parameters
                # --------------------------------------------

                if model_name == "Student-t":

                    record[
                        "student_nu"
                    ] = params_model[0]

                    record[
                        "gennorm_beta"
                    ] = np.nan

                elif model_name == "Generalized Normal":

                    record[
                        "student_nu"
                    ] = np.nan

                    record[
                        "gennorm_beta"
                    ] = params_model[0]

                else:

                    record[
                        "student_nu"
                    ] = np.nan

                    record[
                        "gennorm_beta"
                    ] = np.nan


                model_records.append(
                    record
                )


        # ----------------------------------------------------
        # Retain one standardized residual example per phi
        # ----------------------------------------------------

        if (
            phi0 > 0
            and
            phi0 not in residual_examples
        ):

            residual_examples[
                phi0
            ] = (
                standardized_residual.copy()
            )


# ============================================================
# 23. BUILD DATAFRAMES
# ============================================================

realizations_df = pd.DataFrame(
    realization_records
)

models_df = pd.DataFrame(
    model_records
)

acf_df = pd.DataFrame(
    acf_records
)


# ============================================================
# 24. NUMERICAL SUMMARY BY phi0
# ============================================================

summary_df = (
    realizations_df
    .groupby("phi0")
    .agg(
        n_realizations=(
            "realization",
            "size"
        ),

        epsilon_mean=(
            "epsilon_G",
            "mean"
        ),

        epsilon_std=(
            "epsilon_G",
            "std"
        ),

        skew_mean=(
            "skewness",
            "mean"
        ),

        skew_std=(
            "skewness",
            "std"
        ),

        kurtosis_mean=(
            "pearson_kurtosis",
            "mean"
        ),

        kurtosis_std=(
            "pearson_kurtosis",
            "std"
        ),

        width_mean_mm=(
            "gaussian_width_mm",
            "mean"
        ),

        width_std_mm=(
            "gaussian_width_mm",
            "std"
        ),

        correlation_length_mean_mm=(
            "correlation_length_mm",
            "mean"
        ),

        correlation_length_std_mm=(
            "correlation_length_mm",
            "std"
        )
    )
    .reset_index()
)


# ============================================================
# 25. MODEL-SELECTION SUMMARY
# ============================================================

model_summary_records = []

if len(
    models_df
) > 0:

    winner_rows = (
        models_df
        .loc[
            models_df
            .groupby(
                [
                    "phi0",
                    "realization"
                ]
            )[
                "AIC"
            ]
            .idxmin()
        ]
    )


    for phi0 in sorted(
        winner_rows[
            "phi0"
        ].unique()
    ):

        sub = winner_rows[
            winner_rows[
                "phi0"
            ]
            ==
            phi0
        ]

        n_total = len(
            sub
        )

        for model_name in [
            "Normal",
            "Student-t",
            "Generalized Normal"
        ]:

            n_win = np.sum(
                sub[
                    "model"
                ]
                ==
                model_name
            )

            model_summary_records.append(
                {
                    "phi0":
                        phi0,

                    "model":
                        model_name,

                    "wins":
                        n_win,

                    "total_realizations":
                        n_total,

                    "win_fraction":
                        n_win
                        /
                        n_total
                }
            )


model_summary_df = pd.DataFrame(
    model_summary_records
)


# ============================================================
# 26. MEAN AUTOCORRELATION CURVES
# ============================================================

if len(
    acf_df
) > 0:

    acf_summary_df = (
        acf_df
        .groupby(
            [
                "phi0",
                "lag_index",
                "lag_mm"
            ]
        )
        .agg(
            rho_mean=(
                "rho",
                "mean"
            ),

            rho_std=(
                "rho",
                "std"
            )
        )
        .reset_index()
    )

else:

    acf_summary_df = pd.DataFrame()


# ============================================================
# 27. SAVE TABLES
# ============================================================

summary_df.to_csv(
    "numerical_summary.csv",
    index=False
)

realizations_df.to_csv(
    "realization_statistics.csv",
    index=False
)

models_df.to_csv(
    "model_selection_by_realization.csv",
    index=False
)

model_summary_df.to_csv(
    "model_selection_summary.csv",
    index=False
)

acf_df.to_csv(
    "autocorrelation_curves.csv",
    index=False
)

acf_summary_df.to_csv(
    "autocorrelation_summary.csv",
    index=False
)


# ============================================================
# 28. PRINT MAIN SUMMARY
# ============================================================

print()
print("============================================================")
print("NUMERICAL SUMMARY")
print("============================================================")

print(
    summary_df.to_string(
        index=False
    )
)


print()
print("============================================================")
print("MODEL WIN FRACTIONS — AIC")
print("============================================================")

if len(
    model_summary_df
) > 0:

    print(
        model_summary_df.to_string(
            index=False
        )
    )


# ============================================================
# 29. FIGURE 3
#
# Main statistical result
#
# (a) Gaussian departure
# (b) residual skewness
# (c) residual Pearson kurtosis
# (d) probability-model win fraction
# ============================================================

fig3, axes = plt.subplots(
    2,
    2,
    figsize=(11.5, 8.2)
)


# ------------------------------------------------------------
# (a) epsilon_G
# ------------------------------------------------------------

axes[0,0].errorbar(
    summary_df[
        "phi0"
    ],
    summary_df[
        "epsilon_mean"
    ],
    yerr=summary_df[
        "epsilon_std"
    ],
    marker="o",
    capsize=3,
    linewidth=1.5
)

axes[0,0].set_xlabel(
    r"Perturbation strength $\phi_0$ (rad)"
)

axes[0,0].set_ylabel(
    r"$\langle \epsilon_G\rangle$"
)

axes[0,0].set_title(
    "(a) Departure from Gaussian beam"
)

axes[0,0].grid(
    alpha=0.20
)


# ------------------------------------------------------------
# (b) residual skewness
# ------------------------------------------------------------

axes[0,1].errorbar(
    summary_df[
        "phi0"
    ],
    summary_df[
        "skew_mean"
    ],
    yerr=summary_df[
        "skew_std"
    ],
    marker="o",
    capsize=3,
    linewidth=1.5
)

axes[0,1].axhline(
    0,
    linestyle="--",
    linewidth=1
)

axes[0,1].set_xlabel(
    r"Perturbation strength $\phi_0$ (rad)"
)

axes[0,1].set_ylabel(
    "Residual skewness"
)

axes[0,1].set_title(
    "(b) Residual asymmetry"
)

axes[0,1].grid(
    alpha=0.20
)


# ------------------------------------------------------------
# (c) Pearson kurtosis
# ------------------------------------------------------------

axes[1,0].errorbar(
    summary_df[
        "phi0"
    ],
    summary_df[
        "kurtosis_mean"
    ],
    yerr=summary_df[
        "kurtosis_std"
    ],
    marker="o",
    capsize=3,
    linewidth=1.5
)

axes[1,0].axhline(
    3,
    linestyle="--",
    linewidth=1,
    label="Gaussian reference"
)

axes[1,0].set_xlabel(
    r"Perturbation strength $\phi_0$ (rad)"
)

axes[1,0].set_ylabel(
    "Pearson kurtosis"
)

axes[1,0].set_title(
    "(c) Residual tail structure"
)

axes[1,0].legend(
    frameon=False
)

axes[1,0].grid(
    alpha=0.20
)


# ------------------------------------------------------------
# (d) Model win fractions
# ------------------------------------------------------------

if len(
    model_summary_df
) > 0:

    for model_name in [
        "Normal",
        "Student-t",
        "Generalized Normal"
    ]:

        subset = model_summary_df[
            model_summary_df[
                "model"
            ]
            ==
            model_name
        ]

        axes[1,1].plot(
            subset[
                "phi0"
            ],
            subset[
                "win_fraction"
            ],
            marker="o",
            linewidth=1.5,
            label=model_name
        )


axes[1,1].set_xlabel(
    r"Perturbation strength $\phi_0$ (rad)"
)

axes[1,1].set_ylabel(
    "Fraction of realizations"
)

axes[1,1].set_ylim(
    -0.03,
    1.03
)

axes[1,1].set_title(
    "(d) Best residual model by AIC"
)

axes[1,1].legend(
    frameon=False,
    fontsize=8
)

axes[1,1].grid(
    alpha=0.20
)


fig3.tight_layout()

fig3.savefig(
    "fig3.png",
    dpi=600,
    bbox_inches="tight"
)

fig3.savefig(
    "fig3.pdf",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 30. DIAGNOSTIC RESIDUAL PDF
#
# One representative standardized residual per perturbation.
# ============================================================

if len(
    residual_examples
) > 0:

    plt.figure(
        figsize=(8.2, 5.2)
    )

    bins = np.linspace(
        -6,
        6,
        140
    )


    for (
        phi0,
        residual
    ) in residual_examples.items():

        hist, edges = np.histogram(
            residual,
            bins=bins,
            density=True
        )

        centres = (
            0.5
            * (
                edges[:-1]
                + edges[1:]
            )
        )

        plt.plot(
            centres,
            hist,
            linewidth=1.2,
            label=rf"$\phi_0={phi0:.2f}$"
        )


    z = np.linspace(
        -6,
        6,
        600
    )

    plt.plot(
        z,
        norm.pdf(z),
        linestyle="--",
        linewidth=1.8,
        label="Standard Gaussian"
    )

    plt.xlabel(
        "Standardized Gaussian-fit residual"
    )

    plt.ylabel(
        "Probability density"
    )

    plt.title(
        "Representative residual distributions"
    )

    plt.legend(
        frameon=False,
        fontsize=8
    )

    plt.grid(
        alpha=0.20
    )

    plt.tight_layout()

    plt.savefig(
        "diagnostic_residual_pdfs.png",
        dpi=400,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 31. DIAGNOSTIC CORRELATION-LENGTH FIGURE
#
# This is diagnostic only.
# A cleaner supplementary figure will be generated by the
# dedicated spatial-correlation script.
# ============================================================

if len(
    acf_summary_df
) > 0:

    fig_corr, axes_corr = plt.subplots(
        1,
        2,
        figsize=(11.0, 4.4)
    )


    # --------------------------------------------------------
    # Mean normalized ACF
    # --------------------------------------------------------

    for phi0 in sorted(
        acf_summary_df[
            "phi0"
        ].unique()
    ):

        if np.isclose(
            phi0,
            0
        ):
            continue

        subset = acf_summary_df[
            acf_summary_df[
                "phi0"
            ]
            ==
            phi0
        ]

        axes_corr[0].plot(
            subset[
                "lag_mm"
            ],
            subset[
                "rho_mean"
            ],
            linewidth=1.4,
            label=rf"$\phi_0={phi0:.2f}$"
        )


    axes_corr[0].axhline(
        np.exp(-1),
        linestyle="--",
        linewidth=1,
        label=r"$e^{-1}$"
    )

    axes_corr[0].set_xlim(
        0,
        15
    )

    axes_corr[0].set_xlabel(
        r"Spatial lag $\Delta x$ (mm)"
    )

    axes_corr[0].set_ylabel(
        r"$\rho(\Delta x)$"
    )

    axes_corr[0].set_title(
        "(a) Residual spatial autocorrelation"
    )

    axes_corr[0].legend(
        frameon=False,
        fontsize=8
    )

    axes_corr[0].grid(
        alpha=0.20
    )


    # --------------------------------------------------------
    # Correlation length versus phi0
    # --------------------------------------------------------

    corr_nonzero = summary_df[
        summary_df[
            "phi0"
        ]
        >
        0
    ]

    axes_corr[1].errorbar(
        corr_nonzero[
            "phi0"
        ],
        corr_nonzero[
            "correlation_length_mean_mm"
        ],
        yerr=corr_nonzero[
            "correlation_length_std_mm"
        ],
        marker="o",
        capsize=3,
        linewidth=1.5
    )

    axes_corr[1].set_xlabel(
        r"Perturbation strength $\phi_0$ (rad)"
    )

    axes_corr[1].set_ylabel(
        r"Correlation length $\ell_c$ (mm)"
    )

    axes_corr[1].set_title(
        "(b) Characteristic spatial scale"
    )

    axes_corr[1].grid(
        alpha=0.20
    )


    fig_corr.tight_layout()

    fig_corr.savefig(
        "diagnostic_spatial_correlation.png",
        dpi=400,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 32. STUDENT-t AND GENERALIZED-NORMAL SHAPE PARAMETERS
# ============================================================

if len(
    models_df
) > 0:

    print()
    print("============================================================")
    print("SHAPE PARAMETERS")
    print("============================================================")


    student_df = models_df[
        models_df[
            "model"
        ]
        ==
        "Student-t"
    ]

    gennorm_df = models_df[
        models_df[
            "model"
        ]
        ==
        "Generalized Normal"
    ]


    if len(
        student_df
    ) > 0:

        print()
        print("Student-t degrees of freedom:")

        print(
            student_df
            .groupby(
                "phi0"
            )[
                "student_nu"
            ]
            .agg(
                [
                    "mean",
                    "std",
                    "median"
                ]
            )
        )


    if len(
        gennorm_df
    ) > 0:

        print()
        print("Generalized-normal beta:")

        print(
            gennorm_df
            .groupby(
                "phi0"
            )[
                "gennorm_beta"
            ]
            .agg(
                [
                    "mean",
                    "std",
                    "median"
                ]
            )
        )


# ============================================================
# 33. FINAL OUTPUT SUMMARY
# ============================================================

print()
print("============================================================")
print("FILES GENERATED")
print("============================================================")

print("MAIN FIGURES")
print("  fig2.png")
print("  fig2.pdf")
print("  fig3.png")
print("  fig3.pdf")

print()
print("DIAGNOSTIC FIGURES")
print("  diagnostic_deltaI.png")
print("  diagnostic_residual_pdfs.png")
print("  diagnostic_spatial_correlation.png")

print()
print("DATA")
print("  numerical_summary.csv")
print("  realization_statistics.csv")
print("  model_selection_by_realization.csv")
print("  model_selection_summary.csv")
print("  autocorrelation_curves.csv")
print("  autocorrelation_summary.csv")

print()
print("SOFTWARE")
print(f"  NumPy        : {np.__version__}")
print(f"  SciPy        : {scipy.__version__}")
print(f"  Pandas       : {pd.__version__}")
print(f"  Matplotlib   : {matplotlib.__version__}")

print("============================================================")